In [1]:
import os
from google.colab import drive

# 1. Mount Drive and Setup Paths for eICU
drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/eICU/'

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os
# from google.colab import drive

# 1. Mount Drive and Setup Paths for eICU
# drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/eICU/'

print("Step 1: Loading core eICU administrative and clinical files...")
patient_df = pd.read_csv(base_path + 'patient.csv.gz', compression='gzip')
diagnosis_df = pd.read_csv(base_path + 'diagnosis.csv.gz', compression='gzip')

patient_df.columns = patient_df.columns.str.lower()
diagnosis_df.columns = diagnosis_df.columns.str.lower()

print("\nStep 2: Isolating the cardiac cohort using clinical diagnosis strings...")
cardiac_keywords = 'heart|cardiac|coronary|infarction|failure|arrest|tachycardia|arrhythmia'
cardiac_diagnoses = diagnosis_df[diagnosis_df['diagnosisstring'].str.contains(cardiac_keywords, case=False, na=False)]
cardiac_unit_stay_ids = cardiac_diagnoses['patientunitstayid'].unique()

df_master = patient_df[patient_df['patientunitstayid'].isin(cardiac_unit_stay_ids)].copy()
print(f"Filtered cardiac cohort stays: {df_master.shape[0]}")

print("\nStep 3: Engineering baseline demographic metrics (FIXED)...")
# 1. Force the string clean up
df_master['age_clean'] = df_master['age'].replace('> 89', '90')

# 2. Convert to numeric FIRST, turn any issues into NaN values
df_master['anchor_age'] = pd.to_numeric(df_master['age_clean'], errors='coerce')

# 3. Calculate the median of the actual numbers, fallback to 60 if empty
median_age = df_master['anchor_age'].median()
if pd.isna(median_age):
    median_age = 60

# 4. Fill NaNs with the numeric median value and cast to integer
df_master['anchor_age'] = df_master['anchor_age'].fillna(median_age).astype(int)

# 5. Mask elderly outliers exactly like MIMIC
df_master.loc[df_master['anchor_age'] > 89, 'anchor_age'] = 90

# Map categorical 'gender' string directly to numeric binary flag 'is_male'
df_master['is_male'] = df_master['gender'].map({'Male': 1, 'Female': 0})
# Handle missing values using the numeric mode
gender_mode = df_master['is_male'].mode()[0] if not df_master['is_male'].empty else 1
df_master['is_male'] = df_master['is_male'].fillna(gender_mode).astype(int)

print("\nStep 4: Mapping the classification target label...")
df_master['hospital_expire_flag'] = df_master['hospitaldischargestatus'].apply(lambda x: 1 if str(x).strip().lower() == 'death' else 0)

# Save interim milestone copy to Google Drive
df_master.to_csv(base_path + 'eicu_heart_clinical_master_interim.csv', index=False)

print("\n🎉 PHASE 1 COMPLETE WITH PATCH!")
print(f"Master Matrix Shape: {df_master.shape}")
print(f"Target Label ('hospital_expire_flag') Balance:\n{df_master['hospital_expire_flag'].value_counts()}")

Step 1: Loading core eICU administrative and clinical files...

Step 2: Isolating the cardiac cohort using clinical diagnosis strings...
Filtered cardiac cohort stays: 93700

Step 3: Engineering baseline demographic metrics (FIXED)...

Step 4: Mapping the classification target label...

🎉 PHASE 1 COMPLETE WITH PATCH!
Master Matrix Shape: (93700, 33)
Target Label ('hospital_expire_flag') Balance:
hospital_expire_flag
0    93700
Name: count, dtype: int64


In [3]:
import pandas as pd
base_path = '/content/drive/MyDrive/eICU/'

# Load our interim file
df_master = pd.read_csv(base_path + 'eicu_heart_clinical_master_interim.csv')

print("Unique values found in raw discharge status:")
print(df_master['hospitaldischargestatus'].value_counts(dropna=False))

# Broaden the mapping rule to catch all variations of mortality in eICU
def map_mortality(status):
    status_str = str(status).strip().lower()
    if status_str in ['death', 'expired', 'dead']:
        return 1
    return 0

df_master['hospital_expire_flag'] = df_master['hospitaldischargestatus'].apply(map_mortality)

# Re-save the master with corrected flags
df_master.to_csv(base_path + 'eicu_heart_clinical_master_interim.csv', index=False)

print("\nUpdated Target Label Balance:")
print(df_master['hospital_expire_flag'].value_counts())

Unique values found in raw discharge status:
hospitaldischargestatus
Alive      80594
Expired    12201
NaN          905
Name: count, dtype: int64

Updated Target Label Balance:
hospital_expire_flag
0    81499
1    12201
Name: count, dtype: int64


In [4]:
import pandas as pd
import numpy as np
import os

base_path = '/content/drive/MyDrive/eICU/'
df_master = pd.read_csv(base_path + 'eicu_heart_clinical_master_interim.csv')

target_stay_ids = set(df_master['patientunitstayid'].unique())

vitals_col_mapping = {
    'heartrate': 'vital_mean_heart_rate',
    'systemicsystolic': 'vital_mean_systolic_bp',
    'systemicdiastolic': 'vital_mean_diastolic_bp',
    'sao2': 'vital_mean_spo2',
    'temperature': 'vital_mean_temp'
}

print("Step 1: Initializing streaming chunked parser for vitalPeriodic.csv.gz...")
target_file = base_path + 'vitalPeriodic.csv.gz'

chunk_container = []

vitals_stream = pd.read_csv(
    target_file,
    compression='gzip',
    chunksize=500000,
    usecols=['patientunitstayid', 'heartrate', 'systemicsystolic', 'systemicdiastolic', 'sao2', 'temperature']
)

for i, chunk in enumerate(vitals_stream):
    filtered_chunk = chunk[chunk['patientunitstayid'].isin(target_stay_ids)].copy()

    if not filtered_chunk.empty:
        # CHOSEN FIX: Har chunk ka seedhe mean nikalo taaki MultiIndex ka jhanjhat hi khatam ho jaye
        summary = filtered_chunk.groupby('patientunitstayid').mean()
        chunk_container.append(summary)

    if i % 10 == 0:
        print(f"Streamed and aggregated {i * 500000} time-series rows...")

print("\nStep 2: Combining chunk pieces to calculate final mathematical means...")
# Saare chunk ke means ko ek sath jodo
df_vitals_all = pd.concat(chunk_container)

# Dobara group karke grand mean nikalo jo absolute accurate hoga
df_vitals_final = df_vitals_all.groupby(level=0).mean()

# Columns ko tumhare MIMIC layout ke mutabik rename karo
df_vitals_final = df_vitals_final.rename(columns=vitals_col_mapping).reset_index()

print("\nStep 3: Merging vitals matrix with master cohort matrix...")
# Strict inner/left mapping taaki df_master ka koi data na ude
final_dataset = pd.merge(df_master, df_vitals_final, on='patientunitstayid', how='left')

# Save milestone matrix to disk
final_dataset.to_csv(base_path + 'eicu_heart_clinical_master_with_vitals.csv', index=False)

print("\n🎉 PHASE 2 COMPLETE WITH CLEAN FIX!")
print(f"Merged Dataset Shape: {final_dataset.shape}")
print("Missing Value Counts in Engineered Vitals:")
print(final_dataset[list(vitals_col_mapping.values())].isnull().sum())

Step 1: Initializing streaming chunked parser for vitalPeriodic.csv.gz...
Streamed and aggregated 0 time-series rows...
Streamed and aggregated 5000000 time-series rows...
Streamed and aggregated 10000000 time-series rows...
Streamed and aggregated 15000000 time-series rows...
Streamed and aggregated 20000000 time-series rows...
Streamed and aggregated 25000000 time-series rows...
Streamed and aggregated 30000000 time-series rows...
Streamed and aggregated 35000000 time-series rows...
Streamed and aggregated 40000000 time-series rows...
Streamed and aggregated 45000000 time-series rows...
Streamed and aggregated 50000000 time-series rows...
Streamed and aggregated 55000000 time-series rows...
Streamed and aggregated 60000000 time-series rows...
Streamed and aggregated 65000000 time-series rows...
Streamed and aggregated 70000000 time-series rows...
Streamed and aggregated 75000000 time-series rows...
Streamed and aggregated 80000000 time-series rows...
Streamed and aggregated 85000000 

In [5]:
import pandas as pd
import numpy as np

base_path = '/content/drive/MyDrive/eICU/'
print("Step 1: Loading current master dataset with initial vital signs...")
final_dataset = pd.read_csv(base_path + 'eicu_heart_clinical_master_with_vitals.csv')

# Identify the exact subset of patients who still need vital back-fills
target_stay_ids = set(final_dataset['patientunitstayid'].unique())

# Map nurse charting text labels to your structural feature columns
nurse_mapping = {
    'Heart Rate': 'vital_mean_heart_rate',
    'Non-Invasive BP Systolic': 'vital_mean_systolic_bp',
    'Non-Invasive BP Diastolic': 'vital_mean_diastolic_bp',
    'Temperature (C)': 'vital_mean_temp'
}

print("Step 2: Parsing nurseCharting.csv.gz to extract alternative back-fill values...")
target_file = base_path + 'nurseCharting.csv.gz'
chunk_container = []

# Stream nurse charting in safe chunks
nurse_stream = pd.read_csv(
    target_file,
    compression='gzip',
    chunksize=500000,
    usecols=['patientunitstayid', 'nursingchartcelltypevalname', 'nursingchartvalue']
)

for i, chunk in enumerate(nurse_stream):
    # Filter immediately for target cohort and target numeric vitals
    filtered_chunk = chunk[
        (chunk['patientunitstayid'].isin(target_stay_ids)) &
        (chunk['nursingchartcelltypevalname'].isin(nurse_mapping.keys()))
    ].copy()

    if not filtered_chunk.empty:
        # Force values to numeric types safely, turning errors to NaN
        filtered_chunk['nursingchartvalue'] = pd.to_numeric(filtered_chunk['nursingchartvalue'], errors='coerce')
        filtered_chunk = filtered_chunk.dropna(subset=['nursingchartvalue'])
        chunk_container.append(filtered_chunk)

    if i % 20 == 0:
        print(f"Streamed {i * 500000} nurse charting entries...")

print("\nStep 3: Calculating averages from nurse charting records...")
df_nurse_raw = pd.concat(chunk_container, ignore_index=True)
df_nurse_pivoted = df_nurse_raw.groupby(['patientunitstayid', 'nursingchartcelltypevalname'])['nursingchartvalue'].mean().unstack().reset_index()
df_nurse_pivoted = df_nurse_pivoted.rename(columns=nurse_mapping)

print("\nStep 4: Executing progressive back-fill over missing slots...")
# Set indices for fast alignment matching
final_dataset.set_index('patientunitstayid', inplace=True)
df_nurse_pivoted.set_index('patientunitstayid', inplace=True)

# Combine columns: fill missing values in final_dataset using values from df_nurse_pivoted
for col in nurse_mapping.values():
    if col in df_nurse_pivoted.columns:
        final_dataset[col] = final_dataset[col].fillna(df_nurse_pivoted[col])

# Reset index to return back to normal structure
final_dataset = final_dataset.reset_index()

# Save updated milestone file
final_dataset.to_csv(base_path + 'eicu_heart_clinical_master_with_vitals.csv', index=False)

print("\n🎉 PHASE 2.5 VITAL BACK-FILL COMPLETED SUCCESSFULLY!")
print("New Missing Value Counts in Engineered Vitals:")
vital_cols = ['vital_mean_heart_rate', 'vital_mean_systolic_bp', 'vital_mean_diastolic_bp', 'vital_mean_spo2', 'vital_mean_temp']
print(final_dataset[vital_cols].isnull().sum())

Step 1: Loading current master dataset with initial vital signs...
Step 2: Parsing nurseCharting.csv.gz to extract alternative back-fill values...
Streamed 0 nurse charting entries...
Streamed 10000000 nurse charting entries...
Streamed 20000000 nurse charting entries...
Streamed 30000000 nurse charting entries...
Streamed 40000000 nurse charting entries...
Streamed 50000000 nurse charting entries...
Streamed 60000000 nurse charting entries...
Streamed 70000000 nurse charting entries...
Streamed 80000000 nurse charting entries...
Streamed 90000000 nurse charting entries...
Streamed 100000000 nurse charting entries...
Streamed 110000000 nurse charting entries...
Streamed 120000000 nurse charting entries...
Streamed 130000000 nurse charting entries...
Streamed 140000000 nurse charting entries...
Streamed 150000000 nurse charting entries...

Step 3: Calculating averages from nurse charting records...

Step 4: Executing progressive back-fill over missing slots...

🎉 PHASE 2.5 VITAL BACK-FI

In [9]:
import pandas as pd
import gc

base_path = '/content/drive/MyDrive/eICU/'
final_dataset = pd.read_csv(base_path + 'eicu_heart_clinical_master_with_vitals.csv')
target_stay_ids = set(final_dataset['patientunitstayid'].unique())

print("Step 1: Extracting clinical volume counts from treatment.csv.gz...")
# FIX: usecols hata diya taaki dynamic headers automatic load ho sakein bina crash kiye
treatment_raw = pd.read_csv(
    base_path + 'treatment.csv.gz',
    compression='gzip'
)

# Load hote hi sabse pehle lowercase karo taaki case-sensitivity khatam ho jaye
treatment_raw.columns = treatment_raw.columns.str.lower()

# Sirf wahi do columns rakho jo chahiye, baki memory se clean karo
treatment_df = treatment_raw[['patientunitstayid', 'treatmentstring']].copy()
del treatment_raw
gc.collect()

# Isolate treatments linked to our current heart cohort
cardiac_treatments = treatment_df[treatment_df['patientunitstayid'].isin(target_stay_ids)]

# Calculate row size count per unique ICU stay
proc_counts = cardiac_treatments.groupby('patientunitstayid').size().reset_index(name='visit_procedure_count')

print("Step 2: Consolidating counts with master feature matrix...")
# Purane column ko drop karo agar galti se pehle ka zero wala bacha ho
if 'visit_procedure_count' in final_dataset.columns:
    final_dataset = final_dataset.drop(columns=['visit_procedure_count'])

# Left join ab real data laakar merge karega
final_dataset = pd.merge(final_dataset, proc_counts, on='patientunitstayid', how='left')
final_dataset['visit_procedure_count'] = final_dataset['visit_procedure_count'].fillna(0).astype(int)

# Save progress safely
final_dataset.to_csv(base_path + 'eicu_heart_clinical_master_with_procedures.csv', index=False)

print("\n🎉 PHASE 3 COMPLETE WITH REAL VALUES!")
print(f"Matrix Shape: {final_dataset.shape}")
print(f"Procedure Count Distribution Summary:\n{final_dataset['visit_procedure_count'].describe()}")

Step 1: Extracting clinical volume counts from treatment.csv.gz...
Step 2: Consolidating counts with master feature matrix...

🎉 PHASE 3 COMPLETE WITH REAL VALUES!
Matrix Shape: (93700, 39)
Procedure Count Distribution Summary:
count    93700.00000
mean        31.43174
std        107.95187
min          0.00000
25%          2.00000
50%          7.00000
75%         24.00000
max       7743.00000
Name: visit_procedure_count, dtype: float64


In [10]:
import pandas as pd
import numpy as np
import re

base_path = '/content/drive/MyDrive/eICU/'
print("Step 1: Loading baseline feature matrix...")
final_dataset = pd.read_csv(base_path + 'eicu_heart_clinical_master_with_procedures.csv')

# Load the raw diagnosis table safely
diagnosis_df = pd.read_csv(base_path + 'diagnosis.csv.gz', compression='gzip')
diagnosis_df.columns = diagnosis_df.columns.str.lower()

print("Step 2: Filtering for primary discharge diagnoses...")
# Filter carefully handling case sensitivity inside row values
primary_diag = diagnosis_df[diagnosis_df['diagnosispriority'].astype(str).str.lower() == 'primary'].copy()

# Extract code safely checking if icd9code column exists dynamically
if 'icd9code' in primary_diag.columns:
    primary_diag['icd_clean'] = primary_diag['icd9code'].astype(str).str.split(',').str[0].str.strip()
else:
    # Fallback placeholder if column naming deviates in runtime
    primary_diag['icd_clean'] = '4280'

# Your exact clinical dictionary from your MIMIC tracking pipeline
icd_text_mapping = {
    '40391': 'Hypertensive chronic kidney disease, stage 5 or end-stage renal disease',
    '4019': 'Unspecified essential hypertension',
    '42833': 'Acute on chronic diastolic heart failure',
    '4280': 'Congestive heart failure, unspecified',
    '41401': 'Coronary atherosclerosis of native coronary artery',
    '41071': 'Subendocardial infarction, initial episode of care',
    '412': 'Old myocardial infarction',
    '42731': 'Atrial fibrillation',
    '4241': 'Aortic valve disorders',
    '4240': 'Mitral valve disorders',
    'I10': 'Essential (primary) hypertension',
    'I110': 'Hypertensive heart disease with heart failure',
    'I120': 'Hypertensive chronic kidney disease with stage 5 or end-stage renal disease',
    'I214': 'Acute subendocardial myocardial infarction',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'I480': 'Paroxysmal atrial fibrillation',
    'I481': 'Persistent atrial fibrillation',
    'I482': 'Chronic atrial fibrillation',
    'I509': 'Heart failure, unspecified',
    'I5033': 'Acute on chronic diastolic (congestive) heart failure',
    'I5023': 'Acute on chronic systolic (congestive) heart failure'
}

def get_eicu_description(row):
    code = str(row['icd_clean']).strip()
    if code in icd_text_mapping:
        return icd_text_mapping[code]

    diag_string = str(row['diagnosisstring']).lower()
    if 'congestive heart failure' in diag_string or 'i509' in diag_string:
        return 'Congestive heart failure, unspecified'
    elif 'atrial fibrillation' in diag_string:
        return 'Atrial fibrillation'
    elif 'coronary atherosclerosis' in diag_string:
        return 'Coronary atherosclerosis of native coronary artery'
    elif 'myocardial infarction' in diag_string:
        return 'Subendocardial infarction, initial episode of care'

    if code.startswith('I'):
        return f"ICD-10 Circulatory/Cardiac Condition (Code: {code})"
    return f"ICD-9 Cardiovascular Condition (Code: {code})"

print("Step 3: Compiling textual descriptions...")
primary_diag['icd_description'] = primary_diag.apply(get_eicu_description, axis=1)

# Deduplicate to map exactly one primary condition description per ICU stay anchor
primary_summary = primary_diag.drop_duplicates(subset=['patientunitstayid'])[['patientunitstayid', 'icd_clean', 'icd_description']]
primary_summary = primary_summary.rename(columns={'icd_clean': 'icd_code'})

# Drop duplicates if columns already exist before merge step
cols_to_clean = ['icd_code', 'icd_description']
for col in cols_to_clean:
    if col in final_dataset.columns:
        final_dataset = final_dataset.drop(columns=[col])

# Merge back with the master cohort matrix
final_dataset = pd.merge(final_dataset, primary_summary, on='patientunitstayid', how='left')

print("Step 4: Cleaning administrative columns and preparing for machine learning...")
final_dataset['icd_code'] = final_dataset['icd_code'].fillna('4280')
final_dataset['icd_description'] = final_dataset['icd_description'].fillna('Congestive heart failure, unspecified')

keep_cols = [
    'patientunitstayid', 'icd_code', 'icd_description', 'anchor_age', 'is_male',
    'visit_procedure_count', 'vital_mean_heart_rate', 'vital_mean_systolic_bp',
    'vital_mean_diastolic_bp', 'vital_mean_spo2', 'vital_mean_temp', 'hospital_expire_flag'
]

df_eicu_ready = final_dataset[[col for col in keep_cols if col in final_dataset.columns]].copy()

for col in df_eicu_ready.columns:
    if col not in ['hospital_expire_flag', 'icd_description', 'icd_code'] and df_eicu_ready[col].dtype in [np.float64, np.int64]:
        median_val = df_eicu_ready[col].median()
        if pd.isna(median_val):
            median_val = 0
        df_eicu_ready[col] = df_eicu_ready[col].fillna(median_val)

# Save clean matrix
output_file_path = base_path + 'EICU_CARDIAC_ML_READY.csv'
df_eicu_ready.to_csv(output_file_path, index=False)

print("\n🎉 PHASE 4 CLINICAL TRANSFORMATION COMPLETELY FINISHED!")
print(f"Final eICU Model-Ready Matrix Shape: {df_eicu_ready.shape}")
print("\nTop 5 Most Prevalent Cardiac Diagnoses in your eICU Dataset:")
print(df_eicu_ready['icd_description'].value_counts().head(5))

Step 1: Loading baseline feature matrix...
Step 2: Filtering for primary discharge diagnoses...
Step 3: Compiling textual descriptions...
Step 4: Cleaning administrative columns and preparing for machine learning...

🎉 PHASE 4 CLINICAL TRANSFORMATION COMPLETELY FINISHED!
Final eICU Model-Ready Matrix Shape: (93700, 12)

Top 5 Most Prevalent Cardiac Diagnoses in your eICU Dataset:
icd_description
Congestive heart failure, unspecified                 20824
ICD-9 Cardiovascular Condition (Code: nan)            14382
ICD-9 Cardiovascular Condition (Code: 518.81)          6967
Subendocardial infarction, initial episode of care     4628
ICD-9 Cardiovascular Condition (Code: 038.9)           4532
Name: count, dtype: int64


In [13]:
import pandas as pd
import numpy as np
import re
from google.colab import drive
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# 1. Paths Setup targeting your active eICU directory
base_path = '/content/drive/MyDrive/eICU/'

print("Step 1: Loading your processed, model-ready eICU matrix...")
df = pd.read_csv(base_path + 'EICU_CARDIAC_ML_READY.csv')

# 2. Extract Target Label and Patient Stay Identifiers
y = df['hospital_expire_flag'].astype(int)
groups = df['patientunitstayid']  # Group cross-validation using structural anchor keys

# 3. Handle Features Matrix Separation
cols_to_exclude = ['patientunitstayid', 'icd_code', 'hospital_expire_flag']
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])

print("Step 2: Executing direct One-Hot Encoding on 'icd_description' textual metrics...")
X_encoded = pd.get_dummies(X_raw, columns=['icd_description'], prefix='Dx', prefix_sep='_')

# Clean headers so special characters from clinical text don't crash linear solvers
X_encoded = X_encoded.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

print("Step 3: Verifying and padding remaining sparse blanks via column medians (FIXED)...")
# FIX: Loop aur IF condition hata kar seedhe pooray dataframe par fast vectorised fillna lagaya
X_encoded = X_encoded.fillna(X_encoded.median().fillna(0.0))

# 4. Define Continuous Clinical Features for Normalization
continuous_cols = [
    'anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
    'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
    'vital_mean_spo2', 'vital_mean_temp'
]
continuous_cols = [col for col in continuous_cols if col in X_encoded.columns]

print("Step 4: Splitting train/test splits using GroupKFold via ICU stay identifiers...")
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_encoded, y, groups=groups))

X_train, X_test = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Isolate training specific keys for cross-validation search folds
groups_train = groups.iloc[train_idx]

print("Step 5: Normalizing continuous variables via standard scaler scaling parameters...")
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

# 5. Hyperparameter Space setup for the ElasticNet Solver
param_distributions = {
    'C': np.logspace(-2, 1, 10),
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

cv_strategy = GroupKFold(n_splits=5)

base_lr = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight='balanced',
    max_iter=500,
    random_state=42
)

print("\nStep 6: Executing RandomizedSearchCV optimization layers over ElasticNet hyper-parameters...")
lr_search = RandomizedSearchCV(
    estimator=base_lr,
    param_distributions=param_distributions,
    n_iter=8,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

lr_search.fit(X_train, y_train)

print("\n================ OPTIMIZATION SEARCH RESULTS COMPLETE ================")
print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f} | l1_ratio: {lr_search.best_params_['l1_ratio']:.2f}")

# 6. Evaluate Tuned Estimator on Holdout Test Split
best_model = lr_search.best_estimator_
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

# 7. Compile Patient-Level Prediction Metrics
test_summary = pd.DataFrame({
    'patientunitstayid': df.iloc[test_idx]['patientunitstayid'],
    'y_true': y_test,
    'y_prob': y_prob_tuned
})

patient_res_true = test_summary.groupby('patientunitstayid')['y_true'].max()
patient_res_pred = test_summary.groupby('patientunitstayid')['y_prob'].mean()

# Output evaluation using traditional 0.5 decision boundaries
preds = (patient_res_pred >= 0.5).astype(int)

print("\n--- Final eICU Optimized ElasticNet Evaluation Summary ---")
print(classification_report(patient_res_true, preds))
print(f"Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_true, patient_res_pred):.4f}")

Step 1: Loading your processed, model-ready eICU matrix...
Step 2: Executing direct One-Hot Encoding on 'icd_description' textual metrics...
Step 3: Verifying and padding remaining sparse blanks via column medians (FIXED)...
Step 4: Splitting train/test splits using GroupKFold via ICU stay identifiers...
Step 5: Normalizing continuous variables via standard scaler scaling parameters...

Step 6: Executing RandomizedSearchCV optimization layers over ElasticNet hyper-parameters...


KeyboardInterrupt: 

In [17]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# 1. GPU/CPU Verification
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device execution target: {device}")

base_path = '/content/drive/MyDrive/eICU/'
print("Step 1: Loading model-ready dataset matrices...")
df = pd.read_csv(base_path + 'EICU_CARDIAC_ML_READY.csv')

y = df['hospital_expire_flag'].astype(int).values
groups = df['patientunitstayid'].values

cols_to_drop = ['patientunitstayid', 'icd_code', 'hospital_expire_flag']
X_raw = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

print("Step 2: Encoding icd_description categorical vectors...")
X_encoded = pd.get_dummies(X_raw, columns=['icd_description'], prefix='Dx', prefix_sep='_')
X_encoded = X_encoded.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))
X_encoded = X_encoded.fillna(X_encoded.median().fillna(0.0))

# Dummies lagane ke baad jo booleans bane hain unhe strict numeric integer/float banao
X_encoded = X_encoded.astype(np.float32)

continuous_cols = ['anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate', 'vital_mean_systolic_bp', 'vital_mean_diastolic_bp', 'vital_mean_spo2', 'vital_mean_temp']
continuous_cols = [col for col in continuous_cols if col in X_encoded.columns]

# Split validation sets using 5-Fold Structural CV
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_encoded, y, groups=groups))

X_train_raw, X_test_raw = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y[train_idx], y[test_idx]

print("Step 3: Standardizing structural features via standard scaler...")
scaler = StandardScaler()
X_train_raw[continuous_cols] = scaler.fit_transform(X_train_raw[continuous_cols])
X_test_raw[continuous_cols] = scaler.transform(X_test_raw[continuous_cols])

# FIX: `.astype(np.float32)` force kiya taaki numpy.object_ ka naam-o-nishan mit jaye
X_train_tensor = torch.tensor(X_train_raw.values.astype(np.float32), dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.astype(np.float32), dtype=torch.float32).unsqueeze(1).to(device)
X_test_tensor = torch.tensor(X_test_raw.values.astype(np.float32), dtype=torch.float32).to(device)

# 2. Build Logistic Network Architecture
class PyTorchLogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(PyTorchLogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

model = PyTorchLogisticRegression(X_train_tensor.shape[1]).to(device)

pos_weight_val = (len(y_train) - sum(y_train)) / sum(y_train)
pos_weight = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

print("\nStep 4: Training Logistic Network...")
model.train()
for epoch in range(250):
    optimizer.zero_grad()
    outputs = model(X_train_tensor)

    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch:03d}/250 | Processing Loss Metrics: {loss.item():.4f}")

# 3. Predict and Map back over the test fold partition
model.eval()
with torch.no_grad():
    y_prob_gpu = model(X_test_tensor).cpu().numpy().flatten()

test_summary = pd.DataFrame({
    'patientunitstayid': df.iloc[test_idx]['patientunitstayid'],
    'y_true': y_test,
    'y_prob': y_prob_gpu
})

# Compile clean outcomes aggregates
patient_res_true = test_summary.groupby('patientunitstayid')['y_true'].max()
patient_res_pred = test_summary.groupby('patientunitstayid')['y_prob'].mean()
preds = (patient_res_pred >= 0.5).astype(int)

print("\n================ PYTORCH PIPELINE SUCCESS ================")
print(classification_report(patient_res_true, preds))
print(f"Patient-Level Evaluation ROC-AUC Score: {roc_auc_score(patient_res_true, patient_res_pred):.4f}")

Using device execution target: cpu
Step 1: Loading model-ready dataset matrices...
Step 2: Encoding icd_description categorical vectors...
Step 3: Standardizing structural features via standard scaler...

Step 4: Training Logistic Network...
Epoch 000/250 | Processing Loss Metrics: 1.2561
Epoch 050/250 | Processing Loss Metrics: 1.1449
Epoch 100/250 | Processing Loss Metrics: 1.1212
Epoch 150/250 | Processing Loss Metrics: 1.1114
Epoch 200/250 | Processing Loss Metrics: 1.1057

================ PYTORCH PIPELINE SUCCESS ================
              precision    recall  f1-score   support

           0       0.92      0.89      0.90     16271
           1       0.41      0.51      0.45      2469

    accuracy                           0.84     18740
   macro avg       0.66      0.70      0.68     18740
weighted avg       0.85      0.84      0.85     18740

Patient-Level Evaluation ROC-AUC Score: 0.7875
